# Portfolio Project: Marketing Attribution & Campaign Analytics
## Notebook 02: Data Quality Audit, Cleaning & KPI Engineering

**Role:** Data Analyst  
**Prepared by:** Akshansh Vijay  
**Objective:** This notebook performs systematic data validation checks (duplicates, null values, negative numeric values), formats date fields, engineers primary performance KPIs, and exports the verified dataset for analysis.

### 1. Load Raw Merged Data and Perform Quality Audit

In [1]:
import os
import pandas as pd
import numpy as np

# Resolve path
data_dir = 'data'
if not os.path.exists(data_dir):
    data_dir = os.path.join('..', 'data')

# Load raw merged data
marketing = pd.read_csv(os.path.join(data_dir, 'marketing_raw_merged.csv'))

# 1. Check for missing values
print("Missing Values per Column:")
print(marketing.isnull().sum())
print("-" * 80)

# 2. Check for duplicate rows
duplicates = marketing.duplicated().sum()
print(f"Total Duplicate Rows: {duplicates}")
print("-" * 80)

# 3. Check for negative values in numerical columns
numerical_cols = marketing.select_dtypes(include=['number']).columns
print("Negative Values Check:")
for col in numerical_cols:
    neg_count = (marketing[col] < 0).sum()
    print(f" - {col}: {neg_count} negative value(s)")

Missing Values per Column:
Campaign_ID         0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Impressions         0
Clicks              0
Leads               0
Conversions         0
Revenue             0
Acquisition_Cost    0
ROI                 0
Language            0
Engagement_Score    0
Customer_Segment    0
Date                0
Brand               0
dtype: int64
--------------------------------------------------------------------------------


Total Duplicate Rows: 0
--------------------------------------------------------------------------------
Negative Values Check:
 - Duration: 0 negative value(s)
 - Impressions: 0 negative value(s)
 - Clicks: 0 negative value(s)
 - Leads: 0 negative value(s)
 - Conversions: 0 negative value(s)
 - Revenue: 0 negative value(s)
 - Acquisition_Cost: 0 negative value(s)
 - ROI: 39523 negative value(s)
 - Engagement_Score: 0 negative value(s)


### Rationale on ROI Negatives
During validation, the `ROI` column displays negative values (e.g., `-0.99`). This represents a valid marketing state where a campaign's acquisition expense exceeded its generated revenue (a net financial loss). No rows are dropped or filtered as the rest of the dataset maintains high integrity.

### 2. Format Date Column to Datetime
We convert the `Date` string column to standard datetime objects using `dayfirst=True` to parse `DD-MM-YYYY` dates cleanly.

In [2]:
# Convert to datetime
marketing['Date'] = pd.to_datetime(marketing['Date'], dayfirst=True)
print("Date Column Dtype:", marketing['Date'].dtypes)
print("Date range:", marketing['Date'].min(), "to", marketing['Date'].max())

Date Column Dtype: datetime64[us]
Date range: 2024-07-01 00:00:00 to 2025-06-24 00:00:00


### 3. KPI Engineering & Safe Division-by-Zero Handling
We engineer critical performance metrics:
- **CTR (Click-Through Rate):** `(Clicks / Impressions) * 100`
- **Conversion Rate:** `(Conversions / Clicks) * 100`
- **CPC (Cost Per Click):** `(Acquisition_Cost * Conversions) / Clicks`
- **CPL (Cost Per Lead):** `(Acquisition_Cost * Conversions) / Leads`
- **CPA (Cost Per Acquisition):** `Acquisition_Cost`
- **Revenue Per Conversion:** `Revenue / Conversions`

Dividing by zero clicks, leads, or conversions generates infinite values (`inf`) or undefined numbers (`NaN`). We handle this safely by replacing infinite outputs with `NaN` and filling all `NaN` values with `0.0`.

In [3]:
# Calculate KPIs
marketing['CTR'] = (marketing['Clicks'] / marketing['Impressions']) * 100
marketing['Conversion_Rate'] = (marketing['Conversions'] / marketing['Clicks']) * 100
marketing['CPC'] = (marketing['Acquisition_Cost'] * marketing['Conversions']) / marketing['Clicks']
marketing['CPL'] = (marketing['Acquisition_Cost'] * marketing['Conversions']) / marketing['Leads']
marketing['CPA'] = marketing['Acquisition_Cost']
marketing['Revenue_Per_Conversion'] = marketing['Revenue'] / marketing['Conversions']

kpi_cols = ['CTR', 'Conversion_Rate', 'CPC', 'CPL', 'CPA', 'Revenue_Per_Conversion']

# Safe division-by-zero
marketing[kpi_cols] = marketing[kpi_cols].replace([np.inf, -np.inf], np.nan)
marketing[kpi_cols] = marketing[kpi_cols].fillna(0.0)

print("KPI metrics successfully engineered and infinite/null values filled with 0.0.")
print(marketing[kpi_cols].describe())

KPI metrics successfully engineered and infinite/null values filled with 0.0.


                 CTR  Conversion_Rate            CPC            CPL  \
count  166665.000000    166665.000000  166665.000000  166665.000000   
mean        8.502327        21.945370      69.392686     191.318622   
std         3.753784         8.732106      85.322889     254.339107   
min         1.995921         5.924171       3.538654       6.110204   
25%         5.256101        15.241514      23.014590      58.024076   
50%         8.499456        20.506619      41.675899     109.751443   
75%        11.770451        27.748494      81.617808     220.919820   
max        14.999513        47.869761    1341.071507    5731.627451   

                 CPA  Revenue_Per_Conversion  
count  166665.000000           166665.000000  
mean      376.094881              499.542513  
std       534.939555              173.615813  
min         8.180000              200.000000  
25%       106.730000              349.000000  
50%       208.510000              499.000000  
75%       427.570000           

### 4. Export Cleaned Dataset
Save the fully prepared dataset containing all processed KPIs to `data/marketing_data_cleaned.csv`.

In [4]:
# Export to CSV
output_path = os.path.join(data_dir, 'marketing_data_cleaned.csv')
marketing.to_csv(output_path, index=False)
print(f"Successfully saved clean dataset to: {output_path}")

Successfully saved clean dataset to: ..\data\marketing_data_cleaned.csv
